# LTX-2.5｜Kaggle T4 ×2｜I2V 記憶體穩定快取版

| 模式 Cell | 需要的輸入 | MP4 檔名 |
|---|---|---|
| T2V | Prompt | 自訂 `T2V_RUN_NAME` |
| I2V | Prompt + 一張圖片 | 起始圖片檔名 |
| FLF2V | Prompt + 開頭、結尾圖片 | `開頭_to_結尾.mp4` |

**操作：**執行共用設定 → **只執行其中一個模式 Cell** → 執行生成 → 預覽／下載。變更模式時，只需執行另一個模式 Cell。

FLF2V 雙階段 Workflow 為實驗功能；保留現有 Q4_K_M、雙 GPU、AIMDO 修正與同步音訊配置。

## 1. 安裝套件

In [ ]:

# Install ComfyUI and the custom nodes used by the LTX-2.5 workflow.
# 安裝 LTX-2.5 Workflow 會用到的 ComfyUI 與自訂節點。
import os
import subprocess
import sys
from pathlib import Path
import torch

ROOT = Path('/kaggle/working')
COMFY_DIR = ROOT / 'ComfyUI'
COMFY_REF = 'v0.37.0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

repos = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY_DIR, COMFY_REF),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY_DIR / 'custom_nodes/ComfyUI-GGUF', None),
    ('https://github.com/Lightricks/ComfyUI-LTXVideo.git', COMFY_DIR / 'custom_nodes/ComfyUI-LTXVideo', None),
    ('https://github.com/kijai/ComfyUI-KJNodes.git', COMFY_DIR / 'custom_nodes/ComfyUI-KJNodes', None),
    ('https://github.com/yolain/ComfyUI-Easy-Use.git', COMFY_DIR / 'custom_nodes/ComfyUI-Easy-Use', None),
    ('https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git', COMFY_DIR / 'custom_nodes/workflow-to-api-converter', None),
]

for url, directory, ref in repos:
    if not directory.exists():
        command = ['git', '-c', 'advice.detachedHead=false', 'clone', '--depth', '1', '-q']
        if ref:
            command += ['--branch', ref]
        subprocess.run(command + [url, str(directory)], check=True)

subprocess.run(['git', '-c', 'advice.detachedHead=false', '-C', str(COMFY_DIR), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)

constraint = ROOT / 'ltx25_torch_constraint.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

for _, directory, _ in repos:
    requirement = directory / 'requirements.txt'
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet', 'requests', 'Pillow', 'ipywidgets'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'kornia==0.8.2'], check=True)

legacy = COMFY_DIR / 'custom_nodes/ComfyUI-MultiGPU'
if legacy.is_dir():
    legacy.rename(ROOT / 'ComfyUI-MultiGPU.disabled')

print('ComfyUI installed:', COMFY_DIR)
print('Torch:', torch.__version__)


## 2. 設定環境

In [ ]:
# Import the packages required by the notebook.
# 匯入 Notebook 所需套件。
import importlib.util
import os
import shutil
import subprocess
import time
from pathlib import Path
import ipywidgets as widgets
import requests
import torch
from IPython.display import FileLink, Video, display
from PIL import Image

# Define the Kaggle workspace and ComfyUI paths.
# 設定 Kaggle 與 ComfyUI 的工作路徑。
ROOT = Path('/kaggle/working')
COMFY_DIR = ROOT / 'ComfyUI'
FLOW_REGISTRY_PATH = ROOT / 'ltx25_local_flows.json'
COMFY_LOG = ROOT / 'comfyui_ltx25.log'
COMFY_PORT = 8188

# Store large model weights outside the 20 GB output partition.
# 將大型模型權重放在輸出磁區以外的暫存空間。
SCRATCH_DIR = Path('/tmp/ltx25')
MODEL_STORE = SCRATCH_DIR / 'ltx25_models'

# Create the temporary model directories.
# 建立模型暫存資料夾。
MODEL_STORE.mkdir(parents=True, exist_ok=True)

# Direct Hugging Face downloads and caches to scratch space.
# 將 Hugging Face 下載與快取指定至暫存空間。
os.environ['HF_HOME'] = str(SCRATCH_DIR / 'huggingface')
os.environ['HF_XET_CACHE'] = str(SCRATCH_DIR / 'huggingface' / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Require Kaggle's two-GPU runtime before loading the workflow.
# 載入 Workflow 前，確認 Kaggle 已啟用雙 GPU。
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'

# Display the two detected GPU names.
# 顯示兩張 GPU 的名稱。
for index in range(2):
    # Show the active device name.
    # 顯示目前裝置名稱。
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')


## 3. 模型與 Workflow

首次執行會下載模型及 FLF2V 社群 Workflow；已存在的模型不會重複下載。


In [ ]:
# Fetch only the model files used by this Q4_K_M workflow.
# 只下載 Q4_K_M 雙階段 Workflow 真正需要的模型。
import json
import shutil
import wave
from huggingface_hub import hf_hub_download
from PIL import Image

# Keep the registry compatible with the original conversion engine.
# 保持模型登錄格式與原本的 Workflow 轉換程式相容。
FLOW_REGISTRY = {
    "schema_version": 25,
    "models": {
        "transformers": {"q4km": {"filename": "LTX-2.5-Distilled-Q4_K_M.gguf"}},
        "text_encoder": {"basename": "gemma4-12b-ltx25-w4a8-v2.safetensors"},
        "video_vae": {"filename": "vae/ltx-2.5-video-vae-conv-bf16.safetensors"},
        "audio_vae": {"filename": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
        "spatial_upscaler": {"filename": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
    },
    "workflows": {"t2v_i2v": {"local_name": "LTX25_T2V_I2V.json"}},
}

# Store the downloaded model weights in Kaggle scratch space.
# 將模型權重下載到 Kaggle 暫存空間，減少 working 配額占用。
model_files = [
    ("realrebelai/LTX-2.5_GGUFs", "LTX-2.5-Distilled-Q4_K_M.gguf", "diffusion_models"),
    ("realrebelai/Rebels_w4a8s", "LTX/ENCODERS/gemma4-12b-ltx25-w4a8-v2.safetensors", "text_encoders"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-video-vae-conv-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "vae/ltx-2.5-audio-vae-bf16.safetensors", "vae"),
    ("comfyicu/LTX-2.5", "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors", "latent_upscale_models"),
]

# Link scratch weights into the model folders expected by ComfyUI.
# 透過 symlink 將暫存權重連到 ComfyUI 預期的模型資料夾。
for repo, filename, folder in model_files:
    target = COMFY_DIR / "models" / folder / Path(filename).name
    stored = MODEL_STORE / folder / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    stored.parent.mkdir(parents=True, exist_ok=True)

    # Move previously downloaded weights to scratch without redownloading.
    # 將先前完成下載的權重搬到暫存區，避免重複下載及佔用保存配額。
    if target.is_file() and not target.is_symlink() and target.stat().st_size > 1_000_000:
        if not stored.exists():
            shutil.move(str(target), str(stored))
        elif target.stat().st_size == stored.stat().st_size:
            target.unlink()
        else:
            print("Existing model retained:", target.name)
            continue

    # Download only when the scratch copy is absent.
    # 只有暫存權重不存在時，才執行模型下載。
    if not stored.is_file() or stored.stat().st_size <= 1_000_000:
        stored = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=str(MODEL_STORE / folder)))

    # Replace stale symlinks; never duplicate the large model file.
    # 更新可能失效的 symlink，不另外複製大型權重。
    if target.is_symlink() or target.is_file():
        target.unlink()
    target.symlink_to(stored)
    print("Model:", target.name, "->", stored)

# Expose the same GGUF file to independent GPU loader without duplicating 15 GB of weights.
# 用 symlink 提供 independent GPU loader 載入 GGUF，不重複複製 15 GB 權重。
gguf_path = COMFY_DIR / "models/diffusion_models/LTX-2.5-Distilled-Q4_K_M.gguf"
multigpu_dir = COMFY_DIR / "models/unet_gguf"
multigpu_dir.mkdir(parents=True, exist_ok=True)
multigpu_link = multigpu_dir / gguf_path.name
if multigpu_link.is_symlink() and not multigpu_link.exists():
    multigpu_link.unlink()
if not multigpu_link.exists():
    multigpu_link.symlink_to(gguf_path)

# Obtain the exact original community workflow and its nested subgraphs.
# 取得原本的社群 Workflow，包含其中的巢狀子圖。
workflow_file = hf_hub_download(
    repo_id="realrebelai/LTX-2.5_GGUFs",
    filename="LTX 2.5 GGUF (WORKFLOW).json",
    revision="112436f97aaf99ce13ecb7b7eca7e2f6c128d3ec",
    local_dir=str(SCRATCH_DIR / ".hf_workflow_stage"),
)
ui = json.loads(Path(workflow_file).read_text(encoding="utf-8"))

# Substitute model filenames throughout the UI workflow, including subgraphs.
# 在完整 UI Workflow（含子圖）中替換模型檔名。
replacements = {
    "gguf": gguf_path.name,
    "text_encoder": FLOW_REGISTRY["models"]["text_encoder"]["basename"],
    "video_vae": Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name,
    "audio_vae": Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name,
    "upscaler": Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name,
}

def replace_names(value):
    # Keep this transformation recursive because the workflow contains subgraphs.
    # Workflow 包含巢狀子圖，因此需遞迴處理名稱。
    if isinstance(value, dict):
        return {key: replace_names(item) for key, item in value.items()}
    if isinstance(value, list):
        return [replace_names(item) for item in value]
    if not isinstance(value, str):
        return value
    lower = value.lower()
    if lower.endswith(".gguf") and "ltx" in lower:
        return replacements["gguf"]
    if "gemma4" in lower and lower.endswith(".safetensors") and "enhancer" not in lower:
        return replacements["text_encoder"]
    if "video-vae" in lower and lower.endswith(".safetensors"):
        return replacements["video_vae"]
    if "audio-vae" in lower and lower.endswith(".safetensors"):
        return replacements["audio_vae"]
    if "upscaler" in lower and lower.endswith(".safetensors"):
        return replacements["upscaler"]
    return value

# Prepare valid input placeholders for the workflow converter.
# 準備合法的圖片與音訊預設檔，供 Workflow 轉換器使用。
INPUT_DIR = COMFY_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
Image.new("RGB", (64, 64)).save(INPUT_DIR / "__ltx25_placeholder__.png")
with wave.open(str(INPUT_DIR / "__ltx25_placeholder__.wav"), "wb") as output_audio:
    output_audio.setnchannels(1)
    output_audio.setsampwidth(2)
    output_audio.setframerate(16000)
    output_audio.writeframes(b"\x00\x00" * 16000)

# Replace the sample filenames in nested LoadImage and LoadAudio nodes.
# 將子圖內的範例圖片與音訊名稱改成實際存在的預設檔。
def patch_placeholders(graph):
    for node in graph.get("nodes", []):
        values = node.get("widgets_values")
        if node.get("type") == "LoadImage" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.png"
        if node.get("type") == "LoadAudio" and isinstance(values, list) and values:
            values[0] = "__ltx25_placeholder__.wav"
    for child in graph.get("definitions", {}).get("subgraphs", []):
        patch_placeholders(child)

# Save the patched workflow where the original engine expects it.
# 將 Workflow 存至原本引擎預期的目錄。
ui = replace_names(ui)
patch_placeholders(ui)
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows/LTX25-Kaggle"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
(WORKFLOW_DIR / "LTX25_T2V_I2V.json").write_text(json.dumps(ui, ensure_ascii=False), encoding="utf-8")
FLOW_REGISTRY_PATH.write_text(json.dumps(FLOW_REGISTRY, ensure_ascii=False), encoding="utf-8")
print("Models and workflow ready.")

# Download the two-stage first/last-frame workflow only when missing.
# 僅在缺少雙階段首尾影格 Workflow 時下載一次。
FLF_WORKFLOW_PATH = WORKFLOW_DIR / 'LTX25_FLF2V.json'
if not FLF_WORKFLOW_PATH.is_file():
    # Fetch the community workflow that guides both base and refinement stages.
    # 取得兩階段都加入首尾圖片條件的社群 Workflow。
    flf_url = 'https://raw.githubusercontent.com/UCHIJ/LTX2.5-first_to_last_frame/main/LTX%202.5-first_frame_to_last_frame.json'
    flf_response = requests.get(flf_url, timeout=90)
    flf_response.raise_for_status()
    flf_ui = flf_response.json()

    # Require two image inputs and four keyframe guides before saving.
    # 儲存前確認包含兩張圖片輸入與四個首尾引導節點。
    flf_nodes = list(flf_ui.get('nodes', []))
    for subgraph in flf_ui.get('definitions', {}).get('subgraphs', []):
        flf_nodes.extend(subgraph.get('nodes', []))
    flf_images = [node for node in flf_nodes if node.get('type') == 'LoadImage']
    flf_guides = [node for node in flf_nodes if node.get('type') == 'LTXVAddGuide']
    if len(flf_images) < 2 or len(flf_guides) < 4:
        raise RuntimeError('FLF2V Workflow does not contain two images and two-stage guides.')
    patch_placeholders(flf_ui)
    FLF_WORKFLOW_PATH.write_text(json.dumps(flf_ui, ensure_ascii=False), encoding='utf-8')
print('FLF2V workflow ready:', FLF_WORKFLOW_PATH.name)


# Install the original workflow conversion and output-handling engine locally.
# 在 Kaggle 本機建立原版 Workflow 轉換與輸出引擎。
ENGINE_CODE = '\nimport asyncio\nimport io\nimport math\nimport re\nimport wave\nimport json\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport requests\nfrom PIL import Image\n\nCOMFY_BASE = os.environ.get("COMFY_BASE", "http://127.0.0.1:8188").rstrip("/")\nCOMFY_DIR = Path(os.environ.get("COMFY_DIR", "/kaggle/working/ComfyUI"))\nFLOW_REGISTRY_PATH = Path(os.environ.get("FLOW_REGISTRY_PATH", "/kaggle/working/ltx25_local_flows.json"))\nWORKFLOW_DIR = Path(os.environ.get(\n    "WORKFLOW_DIR",\n    str(COMFY_DIR / "user/default/workflows/LTX25-Kaggle"),\n))\n\nif not FLOW_REGISTRY_PATH.is_file():\n    raise RuntimeError(f"Flow registry missing: {FLOW_REGISTRY_PATH}")\n\nFLOW_REGISTRY = json.loads(FLOW_REGISTRY_PATH.read_text(encoding="utf-8"))\n\nSTACK_SCHEMA_VERSION = int(FLOW_REGISTRY.get("schema_version", 0))\n\nMODEL_FILES = {\n    tier: item["filename"]\n    for tier, item in FLOW_REGISTRY["models"]["transformers"].items()\n}\nPRIMARY_MODEL_TIER = os.environ.get("LTX25_PRIMARY_MODEL_TIER", "q4km").lower()\nFALLBACK_MODEL_TIERS = [\n    x.strip().lower()\n    for x in os.environ.get("LTX25_FALLBACK_MODEL_TIERS", "").split(",")\n    if x.strip()\n]\n\nVAE_TILE_SIZE = int(os.environ.get("LTX25_VAE_TILE_SIZE", "512"))\nVAE_TILE_OVERLAP = int(os.environ.get("LTX25_VAE_TILE_OVERLAP", "64"))\nVAE_TEMPORAL_SIZE = int(os.environ.get("LTX25_VAE_TEMPORAL_SIZE", "32"))\nVAE_TEMPORAL_OVERLAP = int(os.environ.get("LTX25_VAE_TEMPORAL_OVERLAP", "8"))\n\nCOMFY_LOG_PATH = Path(os.environ.get("LTX25_COMFY_LOG", "/kaggle/working/comfyui_ltx25.log"))\nCOMFY_PID_PATH = Path(os.environ.get("LTX25_COMFY_PID", "/kaggle/working/comfyui_ltx25.pid"))\nTEXT_ENCODER = Path(FLOW_REGISTRY["models"]["text_encoder"]["basename"]).name\nVIDEO_VAE = Path(FLOW_REGISTRY["models"]["video_vae"]["filename"]).name\nAUDIO_VAE = Path(FLOW_REGISTRY["models"]["audio_vae"]["filename"]).name\nUPSCALER = Path(FLOW_REGISTRY["models"]["spatial_upscaler"]["filename"]).name\n\nT2V_I2V_WORKFLOW = WORKFLOW_DIR / FLOW_REGISTRY["workflows"]["t2v_i2v"]["local_name"]\nFLF2V_WORKFLOW = WORKFLOW_DIR / "LTX25_FLF2V.json"\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nAUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".ogg"}\nMAX_IMAGE_BYTES = 15 * 1024 * 1024\nMAX_AUDIO_BYTES = 32 * 1024 * 1024\n\nJOBS = {}\nJOBS_LOCK = threading.Lock()\nEXECUTOR = ThreadPoolExecutor(max_workers=1)\n\n\n\ndef job_update(job_key, **fields):\n    # job_key is the dictionary key; fields may safely contain "job_id".\n    with JOBS_LOCK:\n        JOBS.setdefault(job_key, {}).update(fields)\n\n\ndef _comfy_http_error(method, path, response):\n    try:\n        body = response.json()\n        detail = json.dumps(body, ensure_ascii=False, indent=2)\n    except Exception:\n        detail = response.text[:12000]\n    raise RuntimeError(\n        f"ComfyUI {method} {path} failed: HTTP {response.status_code}\\n{detail}"\n    )\n\n\ndef comfy_get(path, **kwargs):\n    response = requests.get(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("GET", path, response)\n    return response\n\n\ndef comfy_post(path, **kwargs):\n    response = requests.post(f"{COMFY_BASE}/{path.lstrip(\'/\')}", **kwargs)\n    if not response.ok:\n        _comfy_http_error("POST", path, response)\n    return response\n\n\ndef get_gguf_loader_choices():\n    # Read only the GPU-selectable GGUF loader exposed by this notebook.\n    # 只檢查本 Notebook 註冊的雙 GPU GGUF Loader。\n    name = \'LTX25GGUFLoaderGPU\'\n    try:\n        spec = comfy_get(f\'/object_info/{name}\', timeout=30).json()[name]\n        return True, spec[\'input\'][\'required\'][\'unet_name\'][0], spec\n    except Exception as exc:\n        return False, [], {\'error\': str(exc)}\n\n\nKEEP_MODELS_RESIDENT = False\n\n\ndef free_comfy():\n    try:\n        comfy_post(\n            "/free",\n            json={"unload_models": True, "free_memory": True},\n            timeout=60,\n        )\n    except Exception:\n        pass\n\n\ndef wait_comfy_ready(timeout=120):\n    deadline = time.time() + timeout\n    last = None\n    while time.time() < deadline:\n        try:\n            response = comfy_get("/system_stats", timeout=3)\n            if response.ok:\n                return True\n        except Exception as exc:\n            last = exc\n        time.sleep(1)\n    raise RuntimeError(f"ComfyUI is not ready: {last}")\n\n\ndef validate_dimensions(width, height):\n    width = int(width)\n    height = int(height)\n    if width < 256 or height < 256:\n        raise ValueError("width and height must both be >= 256")\n    if width % 32 != 0 or height % 32 != 0:\n        raise ValueError("width and height must both be divisible by 32")\n    return width, height\n\n\ndef save_upload_bytes(data, suffix, prefix):\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = f"{prefix}_{uuid.uuid4().hex}{suffix.lower()}"\n    path = input_dir / name\n    path.write_bytes(data)\n    return name, path\n\n\ndef ensure_placeholder_image():\n    """\n    ComfyUI validates LoadImage nodes before execution even when the workflow\n    later disables image conditioning. Some downloaded UI workflows contain\n    stale example filenames such as Test__00139_.png.\n\n    Keep one valid tiny placeholder in ComfyUI/input and patch unused LoadImage\n    nodes to it. The workflow\'s use-image switch still controls conditioning.\n    """\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n\n    name = "__ltx25_placeholder__.png"\n    path = input_dir / name\n\n    if not path.is_file():\n        Image.new("RGB", (64, 64), (0, 0, 0)).save(path, format="PNG")\n\n    return name\n\n\ndef ensure_placeholder_audio():\n    """Create a valid one-second silent WAV for unused LoadAudio validation."""\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    name = "__ltx25_placeholder__.wav"\n    path = input_dir / name\n    if not path.is_file():\n        with wave.open(str(path), "wb") as wf:\n            wf.setnchannels(1)\n            wf.setsampwidth(2)\n            wf.setframerate(16000)\n            wf.writeframes(b"\\x00\\x00" * 16000)\n    return name\n\n\ndef validate_image_bytes(data, suffix):\n    if not data:\n        raise ValueError("Image is empty.")\n    if len(data) > MAX_IMAGE_BYTES:\n        raise ValueError(f"Image exceeds {MAX_IMAGE_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in IMAGE_EXTENSIONS:\n        raise ValueError(f"Unsupported image extension: {suffix}")\n    try:\n        with Image.open(io.BytesIO(data)) as image:\n            image.verify()\n    except Exception as exc:\n        raise ValueError("Invalid/corrupted image.") from exc\n\n\ndef ffprobe_audio_bytes(data, suffix):\n    if not data:\n        raise ValueError("Audio is empty.")\n    if len(data) > MAX_AUDIO_BYTES:\n        raise ValueError(f"Audio exceeds {MAX_AUDIO_BYTES / 2**20:.0f} MiB limit.")\n    if suffix.lower() not in AUDIO_EXTENSIONS:\n        raise ValueError(f"Unsupported audio extension: {suffix}")\n\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:\n        tmp.write(data)\n        path = tmp.name\n\n    try:\n        result = subprocess.run(\n            [\n                "ffprobe", "-v", "error",\n                "-select_streams", "a:0",\n                "-show_entries", "stream=codec_name,profile:format=duration",\n                "-of", "json",\n                path,\n            ],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if result.returncode != 0:\n            raise ValueError("ffprobe could not decode the audio.")\n        payload = json.loads(result.stdout or "{}")\n        streams = payload.get("streams") or []\n        if not streams:\n            raise ValueError("No audio stream found.")\n        duration = float((payload.get("format") or {}).get("duration") or 0)\n        if duration <= 0:\n            raise ValueError("Could not determine audio duration.")\n        return {\n            "codec": streams[0].get("codec_name"),\n            "profile": streams[0].get("profile"),\n            "duration": duration,\n        }\n    finally:\n        try:\n            os.remove(path)\n        except OSError:\n            pass\n\n\ndef _norm_label(value):\n    text = str(value or "").strip().lower().replace("_", " ")\n    text = re.sub(r"[^a-z0-9]+", " ", text)\n    return " ".join(text.split())\n\n\nALIASES = {\n    "prompt": {"prompt", "positive prompt", "prompt positive", "text prompt", "prompt text", "main prompt", "positive text"},\n    "width": {"width", "video width", "output width"},\n    "height": {"height", "video height", "output height"},\n    "seconds": {"duration", "duration in seconds", "seconds", "video duration"},\n    "fps": {"fps", "frame rate", "frames per second"},\n    "seed": {"seed", "noise seed"},\n    "image_strength": {"img strength", "image strength"},\n    "video_vae": {"video vae"},\n    "audio_vae": {"audio vae"},\n    "text_encoder": {"text encoder", "encoder name", "clip name"},\n    "upscaler": {"upscale model", "upscaler", "spatial upscaler"},\n    "text_to_video": {"text to video", "t2v"},\n    "use_image": {"use image input", "use image", "image input"},\n    "use_audio": {"use custom audio", "custom audio", "use audio", "audio input"},\n    "prompt_enhance": {"enhance positive prompt", "prompt enhance", "enhance prompt"},\n    "tile_size": {"tile size"},\n    "tile_overlap": {"tile overlap"},\n    "temporal_size": {"temporal size"},\n    "temporal_overlap": {"temporal overlap"},\n    "model": {"ltx model", "diffusion model", "unet", "unet name", "model checkpoint"},\n}\nALIASES = {k: {_norm_label(x) for x in v} for k, v in ALIASES.items()}\n\n\ndef _matches_field_label(field, label):\n    label = _norm_label(label)\n    if label in ALIASES[field]:\n        return True\n    if field == "prompt" and "prompt" in label:\n        blocked = ("negative", "enhanc", "system", "instruction", "translated", "output")\n        return not any(word in label for word in blocked)\n    return False\n\n\ndef _iter_all_nodes(ui):\n    if not isinstance(ui, dict):\n        return\n    for node in ui.get("nodes", []):\n        if isinstance(node, dict):\n            yield ui, node\n    definitions = ui.get("definitions")\n    subgraphs = definitions.get("subgraphs", []) if isinstance(definitions, dict) else []\n    for subgraph in subgraphs:\n        if isinstance(subgraph, dict):\n            yield from _iter_all_nodes(subgraph)\n\n\ndef _remove_ui_link(owner, link_id):\n    links = owner.get("links")\n    if not isinstance(links, list) or link_id is None:\n        return\n    kept = []\n    for link in links:\n        if isinstance(link, list) and link and link[0] == link_id:\n            continue\n        if isinstance(link, dict) and link.get("id") == link_id:\n            continue\n        kept.append(link)\n    owner["links"] = kept\n\n\ndef _widget_inputs(node):\n    return [x for x in (node.get("inputs") or []) if isinstance(x, dict) and "widget" in x]\n\n\ndef _widget_index(node, input_name):\n    for i, inp in enumerate(_widget_inputs(node)):\n        if inp.get("name") == input_name:\n            return i\n    return None\n\n\ndef _coerce_widget_value(field, value, ui_type):\n    ui_type = str(ui_type or "").upper()\n    if ui_type == "BOOLEAN":\n        return bool(value)\n    if ui_type == "INT":\n        if field == "seconds":\n            return max(1, int(math.ceil(float(value))))\n        return int(round(float(value)))\n    if ui_type == "FLOAT":\n        return float(value)\n    if ui_type in {"STRING", "COMBO"}:\n        return str(value)\n    return value\n\n\ndef _set_exact_widget(owner, node, field, value, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list):\n        return\n    for inp in node.get("inputs") or []:\n        if not isinstance(inp, dict):\n            continue\n        label = _norm_label(inp.get("label") or inp.get("name"))\n        if not _matches_field_label(field, label):\n            continue\n        idx = _widget_index(node, inp.get("name"))\n        if idx is None or idx >= len(values):\n            continue\n        # User-requested controls can be linked to helper nodes in a UI workflow\n        # (the current GGUF workflow links width/height to ResolutionSelector).\n        # Disconnect that incoming UI link so the patched widget value is the\n        # value that survives UI -> API conversion.\n        if field in {\n            "prompt", "width", "height", "seconds", "fps", "seed",\n            "image_strength", "text_to_video", "use_image", "use_audio",\n            "prompt_enhance", "tile_size", "tile_overlap",\n            "temporal_size", "temporal_overlap",\n        } and inp.get("link") is not None:\n            link_id = inp.get("link")\n            _remove_ui_link(owner, link_id)\n            inp["link"] = None\n            stats[f"disconnected_{field}"] = stats.get(f"disconnected_{field}", 0) + 1\n        values[idx] = _coerce_widget_value(field, value, inp.get("type"))\n        stats[field] = stats.get(field, 0) + 1\n\n\ndef _patch_primitives(node, values_by_field, stats):\n    values = node.get("widgets_values")\n    if not isinstance(values, list) or not values:\n        return\n    node_type = str(node.get("type") or "")\n    title = _norm_label(node.get("title"))\n    if node_type in {"PrimitiveString", "PrimitiveStringMultiline"} and _matches_field_label("prompt", title):\n        values[0] = values_by_field["prompt"]\n        stats["prompt"] = stats.get("prompt", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["seconds"]:\n        values[0] = int(math.ceil(values_by_field["seconds"])) if node_type == "PrimitiveInt" else float(values_by_field["seconds"])\n        stats["seconds"] = stats.get("seconds", 0) + 1\n    elif node_type in {"PrimitiveFloat", "PrimitiveInt"} and title in ALIASES["fps"]:\n        values[0] = int(values_by_field["fps"]) if node_type == "PrimitiveInt" else float(values_by_field["fps"])\n        stats["fps"] = stats.get("fps", 0) + 1\n    elif node_type == "PrimitiveInt" and title in ALIASES["seed"]:\n        values[0] = int(values_by_field["seed"])\n        stats["seed"] = stats.get("seed", 0) + 1\n\n\ndef patch_workflow(\n    ui, *, mode, prompt, image_name, last_image_name=None, audio_name, model_file,\n    width, height, seconds, fps, seed, image_strength,\n    tile_size, tile_overlap, temporal_size, temporal_overlap,\n):\n    text_mode = mode in {"t2v", "a2v"}\n    use_image = mode in {"i2v", "ia2v", "flf2v"}\n    use_audio = mode in {"a2v", "ia2v"}\n    values_by_field = {\n        "prompt": prompt, "width": width, "height": height, "seconds": seconds,\n        "fps": fps, "seed": seed, "image_strength": image_strength,\n        "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE,\n        "text_encoder": TEXT_ENCODER, "upscaler": UPSCALER,\n        "text_to_video": text_mode, "use_image": use_image, "use_audio": use_audio,\n        "prompt_enhance": False, "tile_size": tile_size, "tile_overlap": tile_overlap,\n        "temporal_size": temporal_size, "temporal_overlap": temporal_overlap,\n        "model": model_file,\n    }\n    stats = {}\n\n    for owner, node in _iter_all_nodes(ui):\n        node_type = str(node.get("type") or "")\n        widgets = node.get("widgets_values")\n        _patch_primitives(node, values_by_field, stats)\n\n        if node_type == "LoadImage" and isinstance(widgets, list) and widgets:\n            # FLF2V assigns its ending image by the titled LoadImage node.\n            # FLF2V 依照 LoadImage 標題指定結尾圖片。\n            if mode == "flf2v" and "last" in _norm_label(node.get("title")):\n                widgets[0] = last_image_name\n                stats["last_image"] = stats.get("last_image", 0) + 1\n            else:\n                widgets[0] = image_name\n                stats["first_image"] = stats.get("first_image", 0) + 1\n            stats["load_image"] = stats.get("load_image", 0) + 1\n        elif node_type == "LoadAudio" and isinstance(widgets, list) and widgets:\n            widgets[0] = audio_name\n            stats["load_audio"] = stats.get("load_audio", 0) + 1\n        elif node_type in {"UnetLoaderGGUF", "UNETLoader"} and isinstance(widgets, list) and widgets:\n            widgets[0] = model_file\n            stats["model"] = stats.get("model", 0) + 1\n        elif node_type == "LatentUpscaleModelLoader" and isinstance(widgets, list) and widgets:\n            widgets[0] = UPSCALER\n            stats["upscaler"] = stats.get("upscaler", 0) + 1\n\n        for field, value in values_by_field.items():\n            _set_exact_widget(owner, node, field, value, stats)\n\n    # Contract checks: fail here rather than silently generating with stale defaults.\n    required = ["prompt", "width", "height", "seconds", "fps", "seed", "model"]\n    if mode in {"i2v", "ia2v", "flf2v"}:\n        required.append("load_image")\n    if mode == "flf2v":\n        required += ["first_image", "last_image"]\n    missing = [name for name in required if stats.get(name, 0) == 0]\n    if missing:\n        raise RuntimeError(\n            f"Workflow patch contract failed for mode={mode}: missing controls/nodes {missing}. "\n            f"Patch stats={stats}"\n        )\n    return ui, stats\n\n\n_OBJECT_INFO_CACHE = {"at": 0.0, "data": None}\n\n\ndef get_object_info_all(max_age=60):\n    now = time.time()\n    if _OBJECT_INFO_CACHE["data"] is None or now - _OBJECT_INFO_CACHE["at"] > max_age:\n        _OBJECT_INFO_CACHE["data"] = comfy_get("/object_info", timeout=60).json()\n        _OBJECT_INFO_CACHE["at"] = now\n    return _OBJECT_INFO_CACHE["data"]\n\n\ndef convert_workflow(ui_workflow):\n    response = comfy_post("/workflow/convert", json=ui_workflow, timeout=120)\n    payload = response.json()\n    if not isinstance(payload, dict) or not any(\n        isinstance(v, dict) and "class_type" in v for v in payload.values()\n    ):\n        raise RuntimeError(f"Workflow converter returned unexpected payload: {payload}")\n    return payload\n\n\ndef normalize_nested_random_noise(api_workflow):\n    """Use local Gaussian noise that preserves nested video/audio structure."""\n    converted = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "RandomNoise":\n            node["class_type"] = "LTX25NestedRandomNoise"\n            converted += 1\n    if converted == 0:\n        raise RuntimeError("Converted workflow contains no RandomNoise nodes to normalize.")\n    return converted\n\n\ndef normalize_api_workflow_for_gguf(api_workflow, model_file):\n    # Place GGUF diffusion on GPU 0 and encoders/VAEs on GPU 1.\n    # 將 GGUF Diffusion 指定至 GPU 0，文字編碼器與 VAE 指定至 GPU 1。\n    counts = {\'transformer\': 0, \'text_encoder\': 0, \'vae\': 0, \'upscaler\': 0}\n    for node in api_workflow.values():\n        kind = node.get(\'class_type\')\n        inputs = node.get(\'inputs\', {})\n        if kind in {\'UNETLoader\', \'UnetLoaderGGUF\'}:\n            node[\'class_type\'] = \'LTX25GGUFLoaderGPU\'\n            node[\'inputs\'] = {\'unet_name\': model_file, \'device\': \'cuda:0\'}\n            counts[\'transformer\'] += 1\n        elif kind == \'CLIPLoader\':\n            node[\'class_type\'] = \'LTX25CLIPLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'text_encoder\'] += 1\n        elif kind == \'VAELoader\':\n            node[\'class_type\'] = \'LTX25VAELoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'vae\'] += 1\n        elif kind == \'LatentUpscaleModelLoader\':\n            node[\'class_type\'] = \'LTX25UpscalerLoaderGPU\'\n            node[\'inputs\'] = {**inputs, \'device\': \'cuda:1\'}\n            counts[\'upscaler\'] += 1\n    # Reject a workflow that accidentally bypasses either GPU or the two-stage upscaler.\n    # 若 Workflow 略過任一 GPU 或缺少雙階段 Upscaler，直接拒絕執行。\n    if not all(counts.values()):\n        raise RuntimeError(f\'Missing dual-GPU component loaders in converted workflow: {counts}\')\n    print(\'Independent GPU loader mapping:\', counts)\n    return sum(counts.values())\n\n\ndef inject_frozen_external_audio(api_workflow, audio_name, seconds):\n    """Inject existing audio into both stages of the distilled AV workflow.\n\n    Local-only path:\n      LoadAudio -> TrimAudioDuration -> VAEEncodeAudio\n      -> LTXVSetAudioRefTokens -> frozen_audio\n      -> every active LTXVConcatAVLatent audio input\n\n    The normal positive/negative conditioning graph is left unchanged; only the\n    node\'s frozen_audio output is used. For audio-driven modes, both samplers are\n    switched to plain Euler.\n    """\n    if not audio_name:\n        raise RuntimeError("External audio injection requires audio_name.")\n\n    audio_vae_loader = None\n    conditioning_node = None\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict):\n            continue\n        if node.get("class_type") == "VAELoader" and (node.get("inputs") or {}).get("vae_name") == AUDIO_VAE:\n            audio_vae_loader = str(node_id)\n        if conditioning_node is None and node.get("class_type") == "LTXVConditioning":\n            conditioning_node = str(node_id)\n\n    if audio_vae_loader is None:\n        raise RuntimeError(f"Audio VAELoader not found for {AUDIO_VAE!r}.")\n    if conditioning_node is None:\n        raise RuntimeError("LTXVConditioning not found in converted workflow.")\n\n    def fresh_id(base):\n        candidate = base\n        n = 1\n        while candidate in api_workflow:\n            n += 1\n            candidate = f"{base}_{n}"\n        return candidate\n\n    load_id = fresh_id("ltx25_external_audio")\n    trim_id = fresh_id("ltx25_trim_audio")\n    encode_id = fresh_id("ltx25_encode_audio")\n    freeze_id = fresh_id("ltx25_freeze_audio")\n\n    api_workflow[load_id] = {\n        "class_type": "LoadAudio",\n        "inputs": {"audio": audio_name},\n        "_meta": {"title": "LTX25 External Audio"},\n    }\n    api_workflow[trim_id] = {\n        "class_type": "TrimAudioDuration",\n        "inputs": {\n            "audio": [load_id, 0],\n            "start_index": 0.0,\n            "duration": float(seconds),\n        },\n        "_meta": {"title": "LTX25 Trim External Audio"},\n    }\n    api_workflow[encode_id] = {\n        "class_type": "VAEEncodeAudio",\n        "inputs": {\n            "audio": [trim_id, 0],\n            "vae": [audio_vae_loader, 0],\n        },\n        "_meta": {"title": "LTX25 Encode External Audio"},\n    }\n    api_workflow[freeze_id] = {\n        "class_type": "LTXVSetAudioRefTokens",\n        "inputs": {\n            "positive": [conditioning_node, 0],\n            "negative": [conditioning_node, 1],\n            "audio_latent": [encode_id, 0],\n        },\n        "_meta": {"title": "LTX25 Freeze External Audio"},\n    }\n\n    concat_count = 0\n    sampler_count = 0\n    for node in api_workflow.values():\n        if not isinstance(node, dict):\n            continue\n        class_type = node.get("class_type")\n        inputs = node.get("inputs")\n        if not isinstance(inputs, dict):\n            continue\n        if class_type == "LTXVConcatAVLatent":\n            # Official distilled A2V keeps the same encoded audio frozen in both stages.\n            inputs["audio_latent"] = [freeze_id, 2]\n            concat_count += 1\n        elif class_type == "KSamplerSelect":\n            inputs["sampler_name"] = "euler"\n            sampler_count += 1\n\n    if concat_count < 2:\n        raise RuntimeError(\n            "Expected a two-stage AV workflow with at least two LTXVConcatAVLatent nodes; "\n            f"found {concat_count}. The pinned workflow may have changed."\n        )\n    if sampler_count < 2:\n        raise RuntimeError(\n            "Expected at least two KSamplerSelect nodes for the two-stage audio path; "\n            f"found {sampler_count}."\n        )\n\n    return {\n        "frozen_audio_concat_nodes": concat_count,\n        "euler_samplers": sampler_count,\n    }\n\n\ndef _is_connection(value):\n    return (\n        isinstance(value, list) and len(value) == 2\n        and isinstance(value[0], (str, int)) and isinstance(value[1], int)\n    )\n\n\ndef validate_api_workflow(api_workflow):\n    """Structural validation against ComfyUI /object_info without queueing GPU work.\\n\\n    Supports normal required inputs plus newer grouped/dynamic inputs\\n    (for example Autogrow values.a / values.b / ...).\\n    """\n    object_info = get_object_info_all()\n    errors = []\n    for node_id, node in api_workflow.items():\n        if not isinstance(node, dict) or "class_type" not in node:\n            continue\n        class_type = node["class_type"]\n        spec = object_info.get(class_type)\n        if not isinstance(spec, dict):\n            errors.append(f"node {node_id}: class_type {class_type!r} is not registered")\n            continue\n        inputs = node.get("inputs") or {}\n        groups = spec.get("input") or {}\n        required = groups.get("required") or {}\n        optional = groups.get("optional") or {}\n        for name in required:\n            if name in inputs:\n                continue\n\n            # Switch branches are lazy/conditional inputs. Only the branch\n            # selected by `switch` is requested by current ComfyUI.\n            if class_type in {"ComfySwitchNode", "ComfySoftSwitchNode"} and name in {"on_false", "on_true"}:\n                switch_value = inputs.get("switch")\n                if isinstance(switch_value, bool):\n                    selected = "on_true" if switch_value else "on_false"\n                    if name != selected:\n                        continue\n                    # If the statically selected branch is missing, fall through\n                    # and report it. A linked/dynamic switch is deferred.\n                else:\n                    continue\n\n            # ComfyUI\'s newer schema supports grouped/dynamic inputs such as\n            # io.Autogrow.Input("values", ...). The converted API workflow does\n            # not contain a literal "values" key; it contains children such as\n            # "values.a", "values.b", ... instead.\n            #\n            # Treat the required group as present when at least one child key\n            # exists. This is generic and is not hard-coded to ComfyMathExpression.\n            dynamic_prefix = f"{name}."\n            has_dynamic_child = any(\n                isinstance(input_name, str) and input_name.startswith(dynamic_prefix)\n                for input_name in inputs\n            )\n            if has_dynamic_child:\n                continue\n\n            errors.append(f"node {node_id} {class_type}: missing required input {name!r}")\n\n        definitions = dict(optional)\n        definitions.update(required)\n        for name, value in inputs.items():\n            # LoadImage/LoadAudio file lists are dynamic. /object_info may have\n            # been cached before a request uploaded its file, so do not use the\n            # cached combo list as our own file-existence validator. ComfyUI\'s\n            # node validation remains authoritative when /prompt is queued.\n            if (class_type == "LoadAudio" and name == "audio") or (class_type == "LoadImage" and name == "image"):\n                continue\n\n            definition = definitions.get(name)\n            if not isinstance(definition, (list, tuple)) or not definition:\n                continue\n            choices = definition[0]\n            if isinstance(choices, list) and not _is_connection(value):\n                if value not in choices:\n                    preview = choices[:20]\n                    errors.append(\n                        f"node {node_id} {class_type}.{name}: {value!r} not in selectable choices {preview!r}"\n                    )\n    if errors:\n        raise RuntimeError("ComfyUI structural preflight failed:\\n" + "\\n".join(errors[:50]))\n    return True\n\n\ndef queue_prompt(api_workflow):\n    response = comfy_post(\n        "/prompt",\n        json={"prompt": api_workflow, "client_id": str(uuid.uuid4())},\n        timeout=120,\n    )\n    payload = response.json()\n    if payload.get("node_errors"):\n        raise RuntimeError(\n            "ComfyUI rejected workflow:\\n"\n            + json.dumps(payload["node_errors"], ensure_ascii=False, indent=2)\n        )\n    prompt_id = payload.get("prompt_id")\n    if not prompt_id:\n        raise RuntimeError(f"ComfyUI did not return prompt_id: {payload}")\n    return prompt_id\n\n\ndef comfy_crash_diagnostics():\n    parts = []\n\n    pid = None\n    try:\n        if COMFY_PID_PATH.is_file():\n            pid = int(COMFY_PID_PATH.read_text(encoding="utf-8").strip())\n            try:\n                os.kill(pid, 0)\n                parts.append(f"ComfyUI PID {pid} still exists.")\n            except OSError:\n                parts.append(f"ComfyUI PID {pid} is no longer running.")\n    except Exception as exc:\n        parts.append(f"PID check failed: {exc}")\n\n    try:\n        meminfo = {}\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            if ":" in line:\n                key, value = line.split(":", 1)\n                meminfo[key] = value.strip()\n        parts.append(\n            "Host memory after failure: "\n            f"MemAvailable={meminfo.get(\'MemAvailable\', \'?\')}, "\n            f"SwapFree={meminfo.get(\'SwapFree\', \'?\')}."\n        )\n    except Exception:\n        pass\n\n    try:\n        gpu = subprocess.run(\n            [\n                "nvidia-smi",\n                "--query-gpu=memory.total,memory.used,memory.free",\n                "--format=csv,noheader,nounits",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=15,\n        )\n        if gpu.stdout.strip():\n            parts.append("GPU MiB total,used,free: " + gpu.stdout.strip())\n    except Exception:\n        pass\n\n    try:\n        if COMFY_LOG_PATH.is_file():\n            tail = COMFY_LOG_PATH.read_text(errors="replace")[-7000:]\n            parts.append("ComfyUI log tail:\\n" + tail)\n    except Exception as exc:\n        parts.append(f"Could not read ComfyUI log: {exc}")\n\n    return "\\n".join(parts)\n\n\ndef _looks_like_connection_loss(exc):\n    text = str(exc).lower()\n    return (\n        "connection refused" in text\n        or "failed to establish a new connection" in text\n        or "max retries exceeded" in text\n    )\n\n\ndef wait_prompt(prompt_id, timeout=3600):\n    deadline = time.time() + timeout\n    while time.time() < deadline:\n        try:\n            payload = comfy_get(\n                f"/history/{prompt_id}",\n                timeout=60,\n            ).json()\n        except Exception as exc:\n            if _looks_like_connection_loss(exc):\n                raise RuntimeError(\n                    "ComfyUI became unreachable during generation. "\n                    "The ComfyUI process probably exited or was killed by memory pressure.\\n"\n                    + comfy_crash_diagnostics()\n                ) from exc\n            raise\n\n        if prompt_id in payload:\n            history = payload[prompt_id]\n            status = history.get("status", {})\n            if status.get("status_str") == "error":\n                # Show only the failing node and traceback, never megabytes of latent tensors.\n                # 只輸出失敗節點與 traceback，避免將大型 latent tensor 列印到畫面。\n                details = [entry[1] for entry in status.get("messages", []) if entry[0] == "execution_error"]\n                if details:\n                    error = details[-1]\n                    raise RuntimeError(\n                        f"ComfyUI node {error.get(\'node_id\')} ({error.get(\'node_type\')}): "\n                        f"{error.get(\'exception_type\')}: {error.get(\'exception_message\')}\\n"\n                        + "".join(error.get(\'traceback\', [])[-12:])\n                    )\n                raise RuntimeError(f"ComfyUI failed: {status.get(\'messages\', [])[-3:]}")\n            return history\n        time.sleep(3)\n    raise TimeoutError(f"Timed out waiting for ComfyUI prompt {prompt_id}")\n\n\ndef extract_output_item(history):\n    # Prefer video-like outputs. Some nodes use "videos", older wrappers use\n    # "gifs", and newer/custom nodes may expose generic "files".\n    video_exts = {".mp4", ".webm", ".mov", ".mkv", ".gif"}\n    fallback = None\n\n    for output in history.get("outputs", {}).values():\n        if not isinstance(output, dict):\n            continue\n        for key in ("videos", "gifs", "images", "files"):\n            items = output.get(key, [])\n            if not isinstance(items, list):\n                continue\n            for item in items:\n                if not isinstance(item, dict) or not item.get("filename"):\n                    continue\n                suffix = Path(item["filename"]).suffix.lower()\n                if key in {"videos", "gifs"} or suffix in video_exts:\n                    return item\n                if fallback is None:\n                    fallback = item\n\n    if fallback is not None:\n        return fallback\n    raise RuntimeError("Generation completed but no output file was found in ComfyUI history.")\n\n\ndef materialize_output(job_id, item):\n    output_dir = Path("/kaggle/working/ltx25_api_outputs")\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    filename = item.get("filename", f"{job_id}.mp4")\n    suffix = Path(filename).suffix or ".mp4"\n    target = output_dir / f"{job_id}{suffix}"\n\n    response = comfy_get(\n        "/view",\n        params=item,\n        timeout=600,\n    )\n    target.write_bytes(response.content)\n    return target\n\n\ndef mux_original_audio(job_id, video_path, audio_name):\n    """Mux the original uploaded audio back onto the generated video.\n\n    The latent audio drives generation, but the delivered MP4 uses the user\'s\n    original waveform rather than a VAE-decoded reconstruction.\n    """\n    audio_path = COMFY_DIR / "input" / audio_name\n    if not audio_path.is_file():\n        raise RuntimeError(f"Uploaded audio disappeared: {audio_path}")\n\n    output_dir = Path("/kaggle/working/ltx25_api_outputs")\n    target = output_dir / f"{job_id}_with_input_audio.mp4"\n    cmd = [\n        "ffmpeg", "-y", "-v", "error",\n        "-i", str(video_path),\n        "-i", str(audio_path),\n        "-map", "0:v:0",\n        "-map", "1:a:0",\n        "-c:v", "copy",\n        "-c:a", "aac",\n        "-b:a", "192k",\n        "-shortest",\n        str(target),\n    ]\n    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)\n    if result.returncode != 0:\n        raise RuntimeError(f"ffmpeg audio mux failed: {result.stderr[:4000]}")\n    return target\n\n\ndef looks_like_oom(exc):\n    text = str(exc).lower()\n    needles = [\n        "cuda out of memory",\n        "out of memory",\n        "not enough memory",\n        "failed to allocate",\n        "cublas_status_alloc_failed",\n        "cudaerror: out of memory",\n    ]\n    return any(n in text for n in needles)\n\n\ndef choose_workflow(mode):\n    if mode == "flf2v":\n        if not FLF2V_WORKFLOW.is_file():\n            raise RuntimeError(f"FLF2V workflow missing: {FLF2V_WORKFLOW}")\n        return FLF2V_WORKFLOW\n    if mode not in {"t2v", "i2v", "a2v", "ia2v"}:\n        raise RuntimeError(f"Unsupported mode: {mode}")\n    if not T2V_I2V_WORKFLOW.is_file():\n        raise RuntimeError(f"Base workflow missing: {T2V_I2V_WORKFLOW}")\n    return T2V_I2V_WORKFLOW\n\n\ndef prepare_api_workflow(params, tier):\n    mode = params["mode"]\n    model_file = MODEL_FILES[tier]\n    workflow_path = choose_workflow(mode)\n    ui = json.loads(workflow_path.read_text(encoding="utf-8"))\n    ui, patch_stats = patch_workflow(\n        ui,\n        mode=mode,\n        prompt=params["prompt"],\n        image_name=params["image_name"],\n        last_image_name=params.get("last_image_name"),\n        audio_name=params["audio_name"],\n        model_file=model_file,\n        width=params["width"],\n        height=params["height"],\n        seconds=params["seconds"],\n        fps=params["fps"],\n        seed=params["seed"],\n        image_strength=params["image_strength"],\n        tile_size=params["vae_tile_size"],\n        tile_overlap=params["vae_tile_overlap"],\n        temporal_size=params["vae_temporal_size"],\n        temporal_overlap=params["vae_temporal_overlap"],\n    )\n    api_workflow = convert_workflow(ui)\n    nested_noise_nodes = normalize_nested_random_noise(api_workflow)\n    patch_stats["nested_random_noise"] = nested_noise_nodes\n    normalized_loaders = normalize_api_workflow_for_gguf(api_workflow, model_file)\n    if mode == "flf2v":\n        # Both stages must condition on the opening and closing frames.\n        # 兩個階段都必須加入起始與結尾影格條件。\n        guides = [node for node in api_workflow.values() if node.get("class_type") == "LTXVAddGuide"]\n        frame_ids = [node.get("inputs", {}).get("frame_idx") for node in guides]\n        if frame_ids.count(0) < 2 or frame_ids.count(-1) < 2:\n            raise RuntimeError(f"FLF2V requires first/last guides in both stages: {frame_ids}")\n        image_names = {node.get("inputs", {}).get("image") for node in api_workflow.values()\n                       if node.get("class_type") == "LoadImage"}\n        if not {params["image_name"], params["last_image_name"]}.issubset(image_names):\n            raise RuntimeError(f"FLF2V did not bind both images: {image_names}")\n    if normalized_loaders:\n        patch_stats["api_unet_to_gguf"] = normalized_loaders\n    if mode in {"a2v", "ia2v"}:\n        audio_stats = inject_frozen_external_audio(\n            api_workflow, params["audio_name"], params["seconds"]\n        )\n        patch_stats.update(audio_stats)\n    validate_api_workflow(api_workflow)\n    return workflow_path, api_workflow, patch_stats\n\n\ndef run_once(job_id, params, tier):\n    model_file = MODEL_FILES[tier]\n\n    loader_ready, gguf_choices, _ = get_gguf_loader_choices()\n    if not loader_ready:\n        raise RuntimeError("ComfyUI node UnetLoaderGGUF is not registered.")\n    if model_file not in gguf_choices:\n        raise RuntimeError(\n            f"Model exists on disk but UnetLoaderGGUF cannot select {model_file!r}. "\n            f"Choices={gguf_choices}"\n        )\n\n    job_update(job_id, status="preparing", message=f"Patching/validating workflow with {tier}.")\n    workflow_path, api_workflow, patch_stats = prepare_api_workflow(params, tier)\n    job_update(\n        job_id,\n        workflow=str(workflow_path),\n        patch_stats=patch_stats,\n        message=f"Workflow structurally validated with {tier}; queueing ComfyUI.",\n    )\n\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n        time.sleep(1)\n    prompt_id = queue_prompt(api_workflow)\n    job_update(\n        job_id,\n        prompt_id=prompt_id,\n        status="running",\n        message=f"ComfyUI is generating with {tier}.",\n    )\n\n    history = wait_prompt(prompt_id)\n    item = extract_output_item(history)\n    result_path = materialize_output(job_id, item)\n    if params["mode"] in {"a2v", "ia2v"}:\n        result_path = mux_original_audio(job_id, result_path, params["audio_name"])\n    if not KEEP_MODELS_RESIDENT:\n        free_comfy()\n    return result_path\n\n\ndef worker(job_id, params):\n    try:\n        job_update(job_id, status="preparing", message="Preparing local ComfyUI workflow.")\n        # Model policy comes from this Kaggle notebook.\n        configured_chain = [PRIMARY_MODEL_TIER] + [\n            tier for tier in FALLBACK_MODEL_TIERS\n            if tier != PRIMARY_MODEL_TIER\n        ]\n\n        for tier in configured_chain:\n            if tier not in MODEL_FILES:\n                raise RuntimeError(\n                    f"Notebook model policy contains unsupported tier {tier!r}. "\n                    f"Supported: {sorted(MODEL_FILES)}"\n                )\n\n        # Skip configured fallback tiers that have not been downloaded.\n        available_chain = []\n        for tier in configured_chain:\n            model_path = COMFY_DIR / "models/diffusion_models" / MODEL_FILES[tier]\n            if model_path.is_file():\n                available_chain.append(tier)\n\n        if not available_chain:\n            raise RuntimeError(\n                "No configured model tier is downloaded. "\n                f"Notebook policy: {configured_chain}"\n            )\n\n        result = None\n        last_error = None\n        for index, tier in enumerate(available_chain):\n            try:\n                result = run_once(job_id, params, tier)\n                break\n            except Exception as exc:\n                last_error = exc\n                is_last = index == len(available_chain) - 1\n                if is_last or not looks_like_oom(exc):\n                    raise\n\n                next_tier = available_chain[index + 1]\n                free_comfy()\n                job_update(\n                    job_id,\n                    status="retrying",\n                    active_model_tier=next_tier,\n                    message=f"{tier} failed with an OOM-like error; retrying {next_tier}.",\n                    last_tier_error=str(exc),\n                )\n\n        if result is None:\n            raise RuntimeError(f"All model tiers failed. Last error: {last_error}")\n\n        job_update(\n            job_id,\n            status="completed",\n            result_path=str(result),\n            message="Generation completed.",\n        )\n\n    except Exception as exc:\n        if not KEEP_MODELS_RESIDENT:\n            free_comfy()\n        job_update(\n            job_id,\n            status="failed",\n            error=str(exc),\n            message="Generation failed.",\n        )\n\n\ndef _sample_preflight_params(mode):\n    return {\n        "mode": mode,\n        "prompt": "LTX-2.5 structural preflight",\n        "width": 608,\n        "height": 352,\n        "seconds": 3.0,\n        "fps": 24,\n        "seed": 42,\n        "image_strength": 0.7,\n        "vae_tile_size": VAE_TILE_SIZE,\n        "vae_tile_overlap": VAE_TILE_OVERLAP,\n        "vae_temporal_size": VAE_TEMPORAL_SIZE,\n        "vae_temporal_overlap": VAE_TEMPORAL_OVERLAP,\n        "image_name": ensure_placeholder_image(),\n        "audio_name": ensure_placeholder_audio(),\n        "audio_info": None,\n    }\n\n\n_PREFLIGHT_CACHE = {"at": 0.0, "data": None}\n\n\ndef preflight_all_modes(max_age=30):\n    now = time.time()\n    if _PREFLIGHT_CACHE["data"] is not None and now - _PREFLIGHT_CACHE["at"] <= max_age:\n        return _PREFLIGHT_CACHE["data"]\n    result = {}\n    for mode in ("t2v", "i2v", "a2v", "ia2v"):\n        try:\n            workflow_path, _, stats = prepare_api_workflow(\n                _sample_preflight_params(mode), PRIMARY_MODEL_TIER\n            )\n            result[mode] = {\n                "ok": True,\n                "workflow": str(workflow_path),\n                "patch_stats": stats,\n            }\n        except Exception as exc:\n            result[mode] = {\n                "ok": False,\n                "workflow": str(T2V_I2V_WORKFLOW),\n                "error": str(exc),\n            }\n    _PREFLIGHT_CACHE["at"] = now\n    _PREFLIGHT_CACHE["data"] = result\n    return result\n\n\n'

# Keep the helper available to subsequent notebook cells without an external service.
# 將引擎寫入本機檔案，供後續 Cell 使用，不建立外部 API。
ENGINE_PATH = ROOT / "ltx25_kaggle_engine.py"
ENGINE_PATH.write_text(ENGINE_CODE, encoding="utf-8")
print("Original workflow engine + dual-GPU loader patch ready.")


# Create the existing LTX-2.5 nested audio/video noise node.
# 建立原版 LTX-2.5 同步影音 Noise 節點。
LTX25_NOISE_DIR = COMFY_DIR / "custom_nodes/LTX25LocalNoise"
LTX25_NOISE_DIR.mkdir(parents=True, exist_ok=True)
LTX25_NOISE_CODE = 'import numpy as np\nimport torch\nimport comfy.nested_tensor\n\n\ndef _noise_for_tensor(tensor, generator, noise_inds=None):\n    if noise_inds is None:\n        return torch.randn(\n            tensor.size(), dtype=torch.float32, layout=tensor.layout,\n            generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n\n    unique_inds, inverse = np.unique(noise_inds, return_inverse=True)\n    noises = []\n    for i in range(unique_inds[-1] + 1):\n        noise = torch.randn(\n            [1] + list(tensor.size())[1:], dtype=torch.float32,\n            layout=tensor.layout, generator=generator, device="cpu"\n        ).to(dtype=tensor.dtype)\n        if i in unique_inds:\n            noises.append(noise)\n    noises = [noises[i] for i in inverse]\n    return torch.cat(noises, axis=0)\n\n\nclass _LTX25NestedNoise:\n    def __init__(self, seed):\n        self.seed = int(seed)\n\n    def generate_noise(self, input_latent):\n        latent = input_latent["samples"]\n        noise_inds = input_latent.get("batch_index")\n        generator = torch.manual_seed(self.seed)\n\n        if getattr(latent, "is_nested", False):\n            parts = [\n                _noise_for_tensor(part, generator, noise_inds)\n                for part in latent.unbind()\n            ]\n            return comfy.nested_tensor.NestedTensor(parts)\n\n        return _noise_for_tensor(latent, generator, noise_inds)\n\n\nclass LTX25NestedRandomNoise:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "noise_seed": (\n                    "INT",\n                    {\n                        "default": 0,\n                        "min": 0,\n                        "max": 0xFFFFFFFFFFFFFFFF,\n                        "control_after_generate": True,\n                    },\n                )\n            }\n        }\n\n    RETURN_TYPES = ("NOISE",)\n    FUNCTION = "get_noise"\n    CATEGORY = "LTX25/local"\n\n    def get_noise(self, noise_seed):\n        return (_LTX25NestedNoise(noise_seed),)\n\n\nNODE_CLASS_MAPPINGS = {"LTX25NestedRandomNoise": LTX25NestedRandomNoise}\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "LTX25NestedRandomNoise": "LTX25 Nested Random Noise"\n}\n'
(LTX25_NOISE_DIR / "__init__.py").write_text(LTX25_NOISE_CODE, encoding="utf-8")


# Install a small standalone device-placement node pack, not the archived package.
# 安裝獨立的 GPU 裝置選擇節點，不使用封存的套件。
DUAL_NODE_DIR = COMFY_DIR / 'custom_nodes/zz_ltx25_gpu_components'
DUAL_NODE_DIR.mkdir(parents=True, exist_ok=True)
DUAL_NODE_SOURCE = '# Select independent GPUs without archived DisTorch2 or model sharding.\n# 不使用封存的 DisTorch2，以元件方式分配兩張 GPU。\nfrom contextlib import nullcontext\nfrom contextvars import ContextVar\n\n# Access ComfyUI\'s existing model-loading functions.\n# 使用 ComfyUI 原有的模型載入方法。\nimport comfy.model_management as mm\n# Access the original ComfyUI tensor-loading function.\n# 取得 ComfyUI 原本的張量載入函式。\nimport comfy.memory_management as memory_management\n\n# Preserve the original reader for every other model.\n# 其他模型仍沿用原本的讀取方式。\n_original_read_tensor = memory_management.read_tensor_file_slice_into\n\n# Fall back to PyTorch copying for the LTX spatial upscaler only.\n# 僅讓 LTX Spatial Upscaler 改用 PyTorch 複製，避開 AIMDO 傳輸錯誤。\ndef _ltx25_read_tensor(tensor, destination, stream=None, destination2=None):\n    # Locate the checkpoint associated with this tensor, if available.\n    # 取得此張量所屬的權重檔案路徑。\n    tensor_storage = tensor.untyped_storage() if hasattr(tensor, \'untyped_storage\') else None\n    # Check the source metadata attached to an mmap-backed tensor.\n    # 讀取 mmap 張量所附帶的來源資訊。\n    source_path = getattr(tensor_storage, \'_comfy_source_path\', None)\n    # Use the original file handle when source metadata is absent.\n    # 若沒有來源標記，改從原始檔案控制代碼判斷。\n    if source_path is None:\n        # Read the file-slice descriptor used by AIMDO.\n        # 讀取 AIMDO 使用的檔案區段資訊。\n        file_slice = getattr(tensor_storage, \'_comfy_tensor_file_slice\', None)\n        # Derive a filename only when file-slice metadata exists.\n        # 只有在存在檔案區段資訊時，才取得檔案名稱。\n        source_path = getattr(getattr(file_slice, \'file_ref\', None), \'name\', \'\')\n    # Redirect only the known failing LTX-2.5 upscaler weights.\n    # 只對先前出錯的 LTX-2.5 Upscaler 權重採用替代路徑。\n    if \'ltx-2.5-latent-spatial-upscaler\' in str(source_path).lower():\n        # Report the bypass only on its first actual use.\n        # 只在第一次確實使用替代載入路徑時顯示訊息。\n        if not getattr(_ltx25_read_tensor, \'_reported\', False):\n            # Leave one diagnostic line in the ComfyUI log.\n            # 在 ComfyUI Log 保留一行診斷訊息。\n            print(\'[LTX25] Spatial Upscaler: bypassing AIMDO file-to-device transfer\')\n            # Suppress repeated messages for later weight tensors.\n            # 後續張量不再重複印出訊息。\n            _ltx25_read_tensor._reported = True\n        # Let ComfyUI\'s native tensor copy handle the mmap-backed weight.\n        # 交由 ComfyUI 後續的 PyTorch 張量複製流程處理 mmap 權重。\n        return False\n    # Keep AIMDO\'s efficient path for all other models.\n    # 其他模型保留 AIMDO 的低 RAM 載入路徑。\n    return _original_read_tensor(tensor, destination, stream=stream, destination2=destination2)\n\n# Register the targeted fallback before any model is loaded.\n# 在載入任何模型之前，註冊這項局部替代機制。\nmemory_management.read_tensor_file_slice_into = _ltx25_read_tensor\n\nimport folder_paths\nimport torch\nfrom nodes import NODE_CLASS_MAPPINGS as CORE_NODES\n\n# Keep device selection local to a single loader execution.\n# 讓裝置選擇只影響目前執行的 Loader，不改變其他節點的預設行為。\n_active_device = ContextVar(\'ltx25_device\', default=None)\n_original_torch_device = mm.get_torch_device\n_original_text_device = mm.text_encoder_device\n_original_vae_device = mm.vae_device\n\n# Return the selected device only when a loader requests one explicitly.\n# 只有 Loader 明確指定裝置時，才覆寫原本的裝置查詢。\ndef _selected_device(original):\n    selected = _active_device.get()\n    return torch.device(selected) if selected is not None else original()\n\n# Patch the three official device lookup functions without changing memory accounting.\n# 僅覆寫 ComfyUI 三個裝置查詢函式，不改寫記憶體管理程式。\nmm.get_torch_device = lambda: _selected_device(_original_torch_device)\nmm.text_encoder_device = lambda: _selected_device(_original_text_device)\nmm.vae_device = lambda: _selected_device(_original_vae_device)\n\n# Declare the common GPU choices used by every component loader.\n# 統一定義各模型元件使用的 GPU 選項。\nDEVICES = [\'cuda:0\', \'cuda:1\', \'cpu\']\n\n# Convert a new-style ComfyNode output into the legacy tuple format.\n# 將新版 ComfyNode 的 NodeOutput 轉成舊式節點可使用的 tuple。\ndef _as_tuple(result):\n    # Use the wrapped values directly when the node already returns a tuple.\n    # 若原本就回傳 tuple，直接使用，不額外轉換。\n    if isinstance(result, tuple):\n        return result\n    # Newer ComfyUI returns NodeOutput with its values stored in args.\n    # 新版 ComfyUI 的 NodeOutput 會將輸出值保存在 args。\n    if hasattr(result, \'args\'):\n        return tuple(result.args)\n    # Fail early rather than returning an invalid node payload.\n    # 如果輸出格式無法辨識，立即報錯，不讓壞資料流入 sampling。\n    raise TypeError(f\'Unsupported node output type: {type(result).__name__}\')\n\n# Locate an observable load_device field to verify the selected GPU.\n# 查找模型或 patcher 的 load_device，以確認實際裝置。\ndef _load_device(value):\n    # Check the public wrapper and common patcher locations.\n    # 檢查模型本體與常見的 patcher 欄位。\n    for obj in (value, getattr(value, \'patcher\', None), getattr(value, \'model_patcher\', None)):\n        # Skip containers that cannot report a model device.\n        # 忽略無法提供模型裝置的物件。\n        if obj is None:\n            continue\n        # Check explicit inference allocation before other placement fields.\n        # 優先確認模型實際指定的載入裝置。\n        found = getattr(obj, \'load_device\', None)\n        # Return the first explicit device from the patcher.\n        # 回傳第一個有效的載入裝置。\n        if found is not None:\n            return str(found)\n    # Report non-verifiable models rather than inventing a device.\n    # 若無法從物件確認裝置，不推測載入結果。\n    return None\n\n# Use either the legacy FUNCTION API or ComfyNode\'s execute method.\n# 同時支援舊式 FUNCTION 與新版 ComfyNode execute API。\ndef _load_component(name, device, **inputs):\n    # Find the actual core node instead of inheriting incompatible schemas.\n    # 直接取得原始節點，不繼承可能不相容的節點 Schema。\n    node = CORE_NODES.get(name)\n    # Reject missing plugins before sampling or GPU allocation.\n    # 若缺少必要外掛，先報錯，避免開始耗時生成。\n    if node is None:\n        raise RuntimeError(f\'Missing upstream ComfyUI node: {name}\')\n    # Set the target GPU only for this execution.\n    # 將 GPU 指定範圍限制在目前 Loader 的執行過程。\n    token = _active_device.set(device)\n    # Make kernels that use torch.cuda.current_device follow the same GPU.\n    # 讓依賴 torch.cuda.current_device 的 CUDA kernel 也指向正確 GPU。\n    context = torch.cuda.device(device) if device.startswith(\'cuda:\') else nullcontext()\n    try:\n        # Execute original loader code with the selected CUDA device active.\n        # 使用原本的 Loader 程式，同時設定對應 CUDA 裝置。\n        with context:\n            # Prefer the legacy function when the node exposes one.\n            # 如果節點提供舊式 FUNCTION，優先沿用原本載入方法。\n            if hasattr(node, \'FUNCTION\'):\n                # Invoke the original node without passing the custom GPU selector.\n                # 不將自訂 GPU 參數傳入原始 Loader，避免參數不相容。\n                result = getattr(node(), node.FUNCTION)(**inputs)\n            # Handle newer ComfyNode definitions that only provide execute.\n            # 若使用新版 ComfyNode API，改呼叫 execute。\n            elif hasattr(node, \'execute\'):\n                # Use the actual core ComfyNode class method.\n                # 呼叫核心節點提供的新版執行方法。\n                result = node.execute(**inputs)\n            else:\n                # Stop at an incompatible upstream API instead of guessing.\n                # 遇到無法辨識的新版 API 就直接報錯，避免錯誤分配 GPU。\n                raise RuntimeError(f\'Unsupported upstream loader API: {name}\')\n        # Normalize node output and verify its load device if observable.\n        # 整理節點輸出，並在可取得資訊時核對載入裝置。\n        values = _as_tuple(result)\n        # Reject empty output from any model loader.\n        # 拒絕沒有回傳模型的 Loader。\n        if not values:\n            raise RuntimeError(f\'{name} returned no model\')\n        # Compare actual model device with the requested assignment.\n        # 將模型的實際裝置與指定 GPU 比對。\n        actual = _load_device(values[0])\n        # Fail immediately if ComfyUI selected the wrong device.\n        # 若實際裝置與預期不同，立即中止並提示。\n        if actual is not None and actual != device:\n            raise RuntimeError(f\'{name}: requested {device}, actual load_device={actual}\')\n        # Distinguish observed placement from selection-only cases.\n        # 區分已驗證載入位置與僅完成指定裝置的情況。\n        print(f\'[LTX25 dualGPU] {name}: requested={device}, observed={actual or "not exposed"}\')\n        # Return the result in a format compatible with old-style custom nodes.\n        # 使用舊式自訂節點可接收的 tuple 格式回傳。\n        return values\n    finally:\n        # Restore original device behavior for subsequent graph nodes.\n        # 在 Loader 結束後恢復其他節點的預設裝置選擇。\n        _active_device.reset(token)\n\n# Use the core GGUF loader while explicitly selecting GPU 0.\n# 使用 ComfyUI-GGUF 原有 Loader，明確指定 GPU 0。\nclass LTX25GGUFLoaderGPU:\n    # Expose the same model choices that the installed GGUF plugin recognizes.\n    # 使用目前 GGUF 外掛提供的模型清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Declare exact API names required by the existing workflow.\n        # 定義原始 Workflow 需要的 GGUF 模型名稱欄位。\n        return {\'required\': {\'unet_name\': (folder_paths.get_filename_list(\'unet_gguf\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load a quantized LTX transformer through the upstream GGUF node.\n    # 透過 GGUF 外掛載入量化 LTX Transformer。\n    def load(self, unet_name, device):\n        # Reuse the GGUF implementation without any DisTorch2 modifications.\n        # 完全沿用 GGUF Loader，不加入 DisTorch2 的權重修補邏輯。\n        return _load_component(\'UnetLoaderGGUF\', device, unet_name=unet_name)\n\n# Use the standard CLIP loader with a separate GPU allocation.\n# 將 Gemma 文字編碼器分配到另一張 GPU。\nclass LTX25CLIPLoaderGPU:\n    # Keep the original CLIPLoader model-type options.\n    # 保留 ComfyUI 原本的 CLIPLoader 類型選項。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Extend the original choice list with the two CUDA devices.\n        # 在原本的文字模型選單加入雙 GPU 裝置選擇。\n        return {\'required\': {\'clip_name\': (folder_paths.get_filename_list(\'text_encoders\'),), \'type\': (CORE_NODES[\'CLIPLoader\'].INPUT_TYPES()[\'required\'][\'type\'][0],), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'CLIP\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load the selected Gemma encoder without passing a conflicting device keyword.\n    # 不將自訂 device 參數直接傳入原版 CLIPLoader。\n    def load(self, clip_name, type, device):\n        # Leave the core optional device parameter at its default.\n        # 使用原版 Loader 的預設 optional device 參數。\n        return _load_component(\'CLIPLoader\', device, clip_name=clip_name, type=type)\n\n# Use ComfyUI\'s VAE loader for both video and audio weights.\n# Video 與 Audio VAE 都沿用 ComfyUI 的原版模型載入邏輯。\nclass LTX25VAELoaderGPU:\n    # Use the native VAE choice list, including any supported approximations.\n    # 保留 ComfyUI 原本完整的 VAE 清單。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Add explicit GPU selection to the existing VAE filename input.\n        # 在 VAE 檔名欄位加入指定 GPU 選項。\n        return {\'required\': {\'vae_name\': CORE_NODES[\'VAELoader\'].INPUT_TYPES()[\'required\'][\'vae_name\'], \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'VAE\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Load VAE weights with the core loader in the selected CUDA context.\n    # 在指定 CUDA 裝置中呼叫原版 VAE Loader。\n    def load(self, vae_name, device):\n        # Reuse the core VAE loader for the chosen model.\n        # 透過原版 Loader 讀取選定的 VAE 權重。\n        return _load_component(\'VAELoader\', device, vae_name=vae_name)\n\n# Adapt the new-style LatentUpscaleModelLoader without inheriting ComfyNode.\n# 不繼承新版 ComfyNode，避免 LatentUpscaleModelLoader 缺少 INPUT_TYPES。\nclass LTX25UpscalerLoaderGPU:\n    # Supply a legacy-compatible input schema based on public folder paths.\n    # 使用目前 ComfyUI 支援的模型資料夾建立舊式相容 Schema。\n    @classmethod\n    def INPUT_TYPES(cls):\n        # Allow selection of the official LTX-2.5 spatial upscaler.\n        # 提供原生 LTX-2.5 Spatial Upscaler 模型選項。\n        return {\'required\': {\'model_name\': (folder_paths.get_filename_list(\'latent_upscale_models\'),), \'device\': (DEVICES,)}}\n    RETURN_TYPES = (\'LATENT_UPSCALE_MODEL\',)\n    FUNCTION = \'load\'\n    CATEGORY = \'LTX25/dual_gpu\'\n\n    # Call either the new execute method or old FUNCTION method as available.\n    # 依安裝的 ComfyUI 版本呼叫 execute 或原本的 FUNCTION。\n    def load(self, model_name, device):\n        # Keep the original spatial upscaler model-loading semantics.\n        # 保留原始 LatentUpscaleModelLoader 的模型載入邏輯。\n        return _load_component(\'LatentUpscaleModelLoader\', device, model_name=model_name)\n\n# Register the four standalone model loaders.\n# 註冊四種不依賴 MultiGPU 擴充套件的模型 Loader。\nNODE_CLASS_MAPPINGS = {\n    \'LTX25GGUFLoaderGPU\': LTX25GGUFLoaderGPU,\n    \'LTX25CLIPLoaderGPU\': LTX25CLIPLoaderGPU,\n    \'LTX25VAELoaderGPU\': LTX25VAELoaderGPU,\n    \'LTX25UpscalerLoaderGPU\': LTX25UpscalerLoaderGPU,\n}\n\n# Expose readable names in the local ComfyUI server.\n# 在 ComfyUI 本機服務顯示易讀的節點名稱。\nNODE_DISPLAY_NAME_MAPPINGS = {name: name for name in NODE_CLASS_MAPPINGS}\n'
(DUAL_NODE_DIR / '__init__.py').write_text(DUAL_NODE_SOURCE, encoding='utf-8')
print('Standalone GPU loader + targeted upscaler fallback installed:', DUAL_NODE_DIR)


## 4. 啟動 ComfyUI

In [ ]:
# Check and start the persistent ComfyUI service.
# 檢查並啟動可重複使用的 ComfyUI 服務。
import requests
import time

# Check that all required dual-GPU loaders are registered.
# 確認雙 GPU 模型載入節點都已註冊。
def loaders_ready():
    # Use the ComfyUI node registry as the health check.
    # 以 ComfyUI 節點清單確認服務可用。
    try:
        # Verify every required model loader.
        # 逐一驗證必要的模型載入節點。
        for name in ('LTX25GGUFLoaderGPU', 'LTX25CLIPLoaderGPU', 'LTX25VAELoaderGPU', 'LTX25UpscalerLoaderGPU'):
            # Query the running ComfyUI server.
            # 向目前執行的 ComfyUI 查詢節點。
            response = requests.get(f'http://127.0.0.1:{COMFY_PORT}/object_info/{name}', timeout=3)
            # Reject unhealthy or incompatible servers.
            # 拒絕無法回應或缺少節點的服務。
            if not response.ok or name not in response.json():
                # Report a failed health check.
                # 回報健康檢查失敗。
                return False
        # All loaders are available.
        # 全部模型載入節點皆可使用。
        return True
    # A closed port or interrupted server is not ready.
    # 連線失敗代表服務尚未就緒。
    except requests.RequestException:
        # Permit the startup path to recover the service.
        # 允許啟動流程重新建立服務。
        return False

# Avoid retaining large intermediate node outputs between videos.
# 避免不同影片的中間節點結果累積大量 RAM。
STARTUP_FLAGS = [
    '--reserve-vram', '2', '--cache-none', '--fast-disk',
    '--disable-comfy-compiler', '--disable-pinned-memory',
    '--disable-async-offload', '--fp16-unet',
]

# Track whether the running service already uses these settings.
# 記錄目前服務是否已使用相同設定。
CONFIG_ID = 'ltx25-t4x2-i2v-ram-safe-cache-v1'

# Check the existing process before starting another one.
# 啟動新程序前先檢查現有服務。
previous = globals().get('COMFY_PROC')

# Reuse a healthy service started with the same configuration.
# 相同設定且正常運作時，直接沿用目前服務與模型。
reuse_server = (
    previous is not None and previous.poll() is None
    and globals().get('COMFY_CONFIG_ID') == CONFIG_ID
    and loaders_ready()
)

# Only restart to apply changed server settings or recover a dead process.
# 只有需要更新啟動參數或服務中斷時才重新啟動。
if reuse_server:
    # Report that cached models remain available.
    # 顯示模型可沿用，避免不必要的重新載入。
    print('ComfyUI 已在執行，保留模型快取。')
else:
    # Stop the previous notebook-managed server when changing settings.
    # 變更設定時停止原 Notebook 管理的舊服務。
    if previous is not None and previous.poll() is None:
        # Request a graceful termination.
        # 要求舊服務正常結束。
        previous.terminate()
        # Wait until the old process exits.
        # 等待舊程序停止。
        previous.wait(timeout=20)

    # Avoid starting a second server on an occupied port.
    # 避免在已被其他 ComfyUI 占用的連接埠啟動第二個服務。
    if loaders_ready():
        # Require manual cleanup when the process is not notebook-managed.
        # 若現有服務不是 Notebook 管理，先處理連接埠衝突。
        raise RuntimeError('8188 已有 ComfyUI 服務；請停止舊程序，再執行此 Cell。')

    # Run ComfyUI with dynamic VRAM and disk-backed model offloading.
    # 啟用 Dynamic VRAM 與磁碟支援的模型 Offload。
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(COMFY_PORT), *STARTUP_FLAGS]
    # Isolate PyTorch allocator settings for the server.
    # 使用獨立的 PyTorch 記憶體配置設定啟動服務。
    server_env = os.environ.copy()
    # Remove a potentially conflicting legacy allocator setting.
    # 移除可能衝突的舊版記憶體配置參數。
    server_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    # Select the native allocator used by the existing workflow.
    # 沿用現有 Workflow 使用的 native allocator。
    server_env['PYTORCH_ALLOC_CONF'] = 'backend:native'
    # Capture server logs for subsequent crash diagnosis.
    # 儲存 ComfyUI 日誌，方便後續分析中斷原因。
    with COMFY_LOG.open('w', encoding='utf-8') as stream:
        # Run ComfyUI as a persistent subprocess.
        # 以獨立程序常駐啟動 ComfyUI。
        COMFY_PROC = subprocess.Popen(command, cwd=COMFY_DIR, env=server_env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)

    # Wait for all required model loader nodes to be registered.
    # 等待全部雙 GPU 載入節點註冊完成。
    for _ in range(180):
        # Detect startup failures before timing out.
        # 在逾時前先檢查服務是否啟動失敗。
        if COMFY_PROC.poll() is not None:
            # Include recent server logs for diagnosis.
            # 顯示近期日誌，協助找出啟動失敗原因。
            raise RuntimeError(COMFY_LOG.read_text(errors='replace')[-6000:])
        # Stop polling as soon as the server is healthy.
        # 服務就緒後立即結束等待。
        if loaders_ready():
            # Proceed with the running service.
            # 接續使用已啟動的服務。
            break
        # Allow model loader initialization to finish.
        # 等待模型載入節點完成初始化。
        time.sleep(2)
    # The service did not become ready in the expected startup window.
    # 服務未能在指定檢查範圍內完成啟動。
    else:
        # Provide recent logs instead of silently failing.
        # 顯示最近日誌以便排查。
        raise TimeoutError(COMFY_LOG.read_text(errors='replace')[-6000:])

    # Remember this configuration so rerunning the cell will not restart it.
    # 記錄設定，重跑此 Cell 時不重新啟動服務。
    COMFY_CONFIG_ID = CONFIG_ID
    # Report the effective startup policy.
    # 顯示目前生效的記憶體策略。
    print('ComfyUI 已啟動：Dynamic VRAM / cache-none（節點）/ 模型保留。')

# Confirm the two-GPU node configuration is available.
# 確認雙 GPU 節點設定可用。
print('雙 GPU 模型載入節點：正常')


## 5. 共用影片參數

In [ ]:
# Set video dimensions and duration for all three modes.
# 設定三種模式共用的影片尺寸與長度。
SECONDS = 10
# Set the video frame rate.
# 設定影片影格率。
FPS = 24
# Set the generation width and height.
# 設定影片生成的寬度與高度。
WIDTH, HEIGHT = 288, 512

# Set the random seed and input-image guidance strength.
# 設定隨機種子與圖片引導強度。
SEED = 42
# Set image guidance strength for image-conditioned modes.
# 設定圖片生成模式的引導強度。
IMAGE_STRENGTH = 0.7
# Use the previously tested Q4_K_M model.
# 使用先前已測試的 Q4_K_M 模型。
MODEL_TIER = 'q4km' 

# Preserve tiled VAE decoding for limited GPU memory.
# 保留適合有限 GPU 記憶體的 VAE 分塊解碼設定。
VAE_TILE_SIZE, VAE_TILE_OVERLAP = 256, 32
# Set temporal tile dimensions for VAE decoding.
# 設定 VAE 解碼的時間分塊大小。
VAE_TEMPORAL_SIZE, VAE_TEMPORAL_OVERLAP = 16, 4

# Allow filenames to be retained when uploading local images.
# 上傳本機圖片時，保留使用者選取的原始檔名。
def save_selected_image(change, folder):
    # Ignore the empty upload event.
    # 忽略尚未選取檔案的事件。
    if not change['new']:
        # Keep the existing selection unchanged.
        # 保留目前已選取的檔案。
        return None

    # Read either the old or new ipywidgets upload format.
    # 讀取新舊版本 ipywidgets 的上傳資料。
    uploaded = change['new']
    # Normalize the uploaded item format.
    # 將上傳項目整理成統一格式。
    if isinstance(uploaded, dict):
        # Extract the original filename and its content.
        # 取得原始檔名與檔案內容。
        filename, item = next(iter(uploaded.items()))
    else:
        # Read the selected file from the newer widget format.
        # 從新版 widget 格式取得所選檔案。
        item = uploaded[0]
        filename = item['name']

    # Keep the original basename without directory components.
    # 保留原始檔名，排除不必要的目錄路徑。
    filename = Path(filename.replace('\\', '/')).name
    # Save different image roles in separate folders to prevent collisions.
    # 依圖片用途分資料夾儲存，避免同名檔案互相覆蓋。
    destination = ROOT / 'ltx25_images' / folder / filename
    # Create the destination folder on first upload.
    # 第一次上傳時建立對應的資料夾。
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Persist the uploaded image using its original name.
    # 使用原始檔名儲存上傳的圖片。
    destination.write_bytes(bytes(item['content']))
    # Show the selected image's location for the current mode.
    # 顯示目前模式所使用的圖片路徑。
    print('已選取：', destination)
    # Return the saved image path to the mode callback.
    # 將圖片路徑交回目前模式的上傳處理函式。
    return str(destination)

# Explain that only the selected mode cell determines the next generation.
# 說明下一段影片只會使用最後執行的模式 Cell。
print('影片設定完成；接著只執行 T2V、I2V、FLF2V 其中一格。')

## 6A. T2V｜文字生成影片（不用圖片）

In [ ]:
# Define the prompt specifically for text-to-video generation.
# 設定只供文字生成影片使用的提示詞。
T2V_PROMPT = '''
電影感多視角鏡頭序列，總時長 10 秒。主角始終是同一位烏黑直長髮、氣質溫柔的韓國年輕女大學生，身穿米白色寬鬆針織開襟衫，肩背帆布托特包；人物臉部特徵、服裝與髮型保持一致。
場景是清晨的櫻花林蔭大道，兩側盛開粉紅櫻花，淡淡晨霧籠罩，柔和金色陽光穿過樹梢，花瓣隨微風緩緩飄落。
0至3秒（側身遠景）：她靜靜站在櫻花大道旁，微微側身眺望遠方，鏡頭平穩取景。
3至6秒（正面中景）：鏡頭流暢移動到她的正前方並緩緩推近，她轉向鏡頭，神情自然柔和。
6至8秒（臉部特寫）：鏡頭繼續推近，她輕輕抬眼直視鏡頭，露出甜美而溫暖的微笑。
8至10秒（微笑與眨眼）：維持正面特寫與眼神接觸，她在最後一秒俏皮地單眼眨眼（wink），隨即保持微笑直到最後一格；微風吹動髮絲。
35mm 底片美學、連貫自然運鏡、浪漫淺景深、柔和粉彩色調、自然皮膚紋理與流暢動作。
'''.strip()

# Give text-to-video an explicit clip name because it has no source image.
# T2V 沒有來源圖片，因此使用自訂影片名稱。
T2V_RUN_NAME = 'clip01'

# Activate T2V without requiring an image upload.
# 啟用 T2V，不需要上傳任何圖片。
ACTIVE_JOB = {'mode': 't2v', 'prompt': T2V_PROMPT, 'run_name': T2V_RUN_NAME}

# Confirm which mode the generation cell will use.
# 確認生成 Cell 將使用的模式。
print('目前模式：T2V | 輸出名稱：', T2V_RUN_NAME)

## 6B. I2V｜上傳一張圖片

In [ ]:
# Define the prompt specifically for image-to-video generation.
# 設定只供單張圖片生成影片使用的提示詞。
I2V_PROMPT = 'The person in the photograph begins to walk naturally, smooth cinematic camera movement.'

# Optionally reuse a photo from Kaggle working or an attached dataset.
# 可直接使用 Kaggle working 或附加資料集中的既有照片。
I2V_IMAGE_PATH = globals().get('I2V_IMAGE_PATH')

# Activate I2V while retaining its previously uploaded photo, if any.
# 啟用 I2V，並保留先前上傳的照片路徑。
ACTIVE_JOB = {'mode': 'i2v', 'prompt': I2V_PROMPT, 'first_path': I2V_IMAGE_PATH}

# Keep this mode's selection separate from later mode changes.
# 將本模式的上傳狀態與後續模式切換隔離。
i2v_job = ACTIVE_JOB

# Update this I2V job when a local photo is selected.
# 選取本機照片後，更新此 I2V 工作設定。
def on_i2v_upload(change):
    # Update the reusable I2V image path.
    # 更新下次也可沿用的 I2V 圖片路徑。
    global I2V_IMAGE_PATH
    # Save the image while preserving its original filename.
    # 保留原始檔名並儲存圖片。
    selected = save_selected_image(change, 'i2v')
    # Update the selected I2V job only when a file was uploaded.
    # 只有成功選取圖片時才更新 I2V 工作設定。
    if selected:
        # Store the image for the next I2V run.
        # 儲存供下一次 I2V 使用的圖片路徑。
        I2V_IMAGE_PATH = selected
        i2v_job['first_path'] = selected
        # Show the clip name derived from the input image filename.
        # 顯示由來源圖片檔名產生的影片名稱。
        print('影片名稱：', Path(selected).stem + '.mp4')

# Provide an upload picker inside the I2V mode cell.
# 直接在 I2V 模式 Cell 顯示本機照片上傳按鈕。
i2v_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇 I2V 照片')
# Handle future uploads without rerunning this cell.
# 不必重新執行 Cell，也能處理後續選取的照片。
i2v_upload.observe(on_i2v_upload, names='value')
# Display the control for selecting a local image.
# 顯示本機圖片選取按鈕。
display(i2v_upload)
# Show the active mode and current image if available.
# 顯示目前模式與已指定的圖片。
print('目前模式：I2V | 圖片：', I2V_IMAGE_PATH or '請上傳照片')

## 6C. FLF2V｜分別上傳開頭與結尾圖片

In [ ]:
# Define the prompt specifically for first/last-frame generation.
# 設定只供首尾影格影片生成使用的提示詞。
FLF2V_PROMPT = 'Smooth cinematic motion transitioning naturally between the opening image and ending image.'

# Reuse prior selections or enter existing Kaggle image paths here.
# 沿用先前選取的圖片，或在此填入 Kaggle 中既有的圖片路徑。
FLF2V_FIRST_IMAGE_PATH = globals().get('FLF2V_FIRST_IMAGE_PATH')
FLF2V_LAST_IMAGE_PATH = globals().get('FLF2V_LAST_IMAGE_PATH')

# Activate FLF2V with two independent endpoint images.
# 啟用 FLF2V，分別使用開頭與結尾圖片。
ACTIVE_JOB = {
    'mode': 'flf2v',
    'prompt': FLF2V_PROMPT,
    'first_path': FLF2V_FIRST_IMAGE_PATH,
    'last_path': FLF2V_LAST_IMAGE_PATH,
}

# Keep the selected FLF2V job separate from the other modes.
# 讓 FLF2V 的檔案選取狀態與其他模式互不影響。
flf2v_job = ACTIVE_JOB

# Save the uploaded image under its correct endpoint role.
# 依開頭或結尾用途儲存選取的圖片。
def on_flf2v_upload(change, role):
    # Update the reusable first and last image paths.
    # 更新可供下次使用的首尾圖片路徑。
    global FLF2V_FIRST_IMAGE_PATH, FLF2V_LAST_IMAGE_PATH
    # Save with the original image name in a separate endpoint folder.
    # 依用途分資料夾保留所選圖片的原始檔名。
    selected = save_selected_image(change, 'flf_first' if role == 'first' else 'flf_last')
    # Keep the prior selection if no file was uploaded.
    # 沒有選取檔案時保留原本的圖片路徑。
    if not selected:
        # Ignore empty upload changes.
        # 忽略沒有圖片的上傳變更。
        return
    # Update only the endpoint that the user selected.
    # 只更新使用者本次選取的起始或結尾圖片。
    if role == 'first':
        # Save the opening image path separately.
        # 獨立儲存開頭圖片路徑。
        FLF2V_FIRST_IMAGE_PATH = selected
        flf2v_job['first_path'] = selected
    else:
        # Save the ending image path separately.
        # 獨立儲存結尾圖片路徑。
        FLF2V_LAST_IMAGE_PATH = selected
        flf2v_job['last_path'] = selected
    # Show both filenames so the user can confirm the endpoints.
    # 顯示兩個檔名，方便確認起點與終點。
    print('開頭：', Path(flf2v_job['first_path']).name if flf2v_job['first_path'] else '未選取')
    print('結尾：', Path(flf2v_job['last_path']).name if flf2v_job['last_path'] else '未選取')

# Create two separate local-file upload buttons.
# 建立兩個獨立的本機圖片上傳按鈕。
flf_first_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇開頭照片')
flf_last_upload = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='選擇結尾照片')
# Connect each button to its corresponding endpoint.
# 將兩個按鈕分別連接到正確的影格位置。
flf_first_upload.observe(lambda change: on_flf2v_upload(change, 'first'), names='value')
flf_last_upload.observe(lambda change: on_flf2v_upload(change, 'last'), names='value')
# Display both controls in this FLF2V mode cell.
# 在 FLF2V 模式 Cell 中同時顯示兩個上傳按鈕。
display(flf_first_upload, flf_last_upload)
# Confirm the active mode.
# 顯示已啟用的生成模式。
print('目前模式：FLF2V')

## 7. 生成影片

In [ ]:
# Load the original two-stage LTX-2.5 workflow engine.
# 載入原本的 LTX-2.5 雙階段 Workflow 引擎。
spec = importlib.util.spec_from_file_location('ltx25_kaggle_engine', ENGINE_PATH)
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

# Retain models across completed generations; still allow memory-pressure offloading.
# 保留跨次生成的模型快取，並允許記憶體壓力下的自動 Offload。
engine.KEEP_MODELS_RESIDENT = True

# Read an immutable snapshot of the last executed mode cell.
# 取得最後執行的模式 Cell 設定，避免生成中途被上傳操作更動。
if 'ACTIVE_JOB' not in globals():
    # Select one mode cell before starting generation.
    # 開始生成前，必須先執行其中一個模式 Cell。
    raise ValueError('請先執行 T2V、I2V 或 FLF2V 的模式 Cell。')

# Copy the active job to prevent later widget events from changing this run.
# 複製目前的工作設定，避免後續上傳事件改動本次生成。
job = ACTIVE_JOB.copy()
# Read the selected mode and its independent prompt.
# 讀取目前模式及其專屬 Prompt。
MODE = job['mode']
PROMPT = job['prompt']

# Keep a placeholder image for text-only generation.
# 純文字模式使用預設空白圖片。
image_name = engine.ensure_placeholder_image()
last_image_name = None

# Require the opening image for either image-conditioned mode.
# 只要使用圖片生成模式，就必須先取得開頭照片。
if MODE in ('i2v', 'flf2v'):
    # Get the selected first image path.
    # 取得選取的起始圖片路徑。
    first_path = Path(job['first_path']) if job.get('first_path') else None
    # Reject missing images before starting GPU generation.
    # 在 GPU 開始生成前拒絕缺少圖片的設定。
    if first_path is None or not first_path.is_file():
        # Instruct the user to upload the opening photo.
        # 提示使用者先上傳起始照片。
        raise ValueError(f'{MODE.upper()} 請先在模式 Cell 選取開頭照片。')
    # Derive the I2V output name from the original image filename.
    # I2V 使用來源圖片的檔名作為輸出影片名稱。
    RUN_NAME = first_path.stem
    # Upload the first frame to the ComfyUI input folder.
    # 將開頭照片放入 ComfyUI 的輸入資料夾。
    image_name, _ = engine.save_upload_bytes(first_path.read_bytes(), first_path.suffix, RUN_NAME)
else:
    # Use the explicit T2V clip name when no image is supplied.
    # T2V 沒有來源圖片，使用自訂的影片名稱。
    RUN_NAME = job['run_name']

# Require a distinct ending image for first/last-frame generation.
# 首尾影格模式需額外指定結尾照片。
if MODE == 'flf2v':
    # Read the ending image selected by the second upload button.
    # 讀取第二個按鈕選取的結尾照片。
    last_path = Path(job['last_path']) if job.get('last_path') else None
    # Reject an unset or missing final image.
    # 沒有指定或找不到結尾照片時立即報錯。
    if last_path is None or not last_path.is_file():
        # Explain which input is missing.
        # 提示缺少結尾照片。
        raise ValueError('FLF2V 請在模式 Cell 選取結尾照片。')
    # Name the clip using both source filenames.
    # 以開頭與結尾圖片的檔名組合輸出影片名稱。
    RUN_NAME = f'{first_path.stem}_to_{last_path.stem}'
    # Upload the final frame independently of the first frame.
    # 獨立上傳結尾照片，避免覆蓋開頭照片。
    last_image_name, _ = engine.save_upload_bytes(last_path.read_bytes(), last_path.suffix, f'{RUN_NAME}_last')

# Construct output locations after the selected mode and filenames are known.
# 確定模式及檔名後，再建立輸出影片與影格的路徑。
OUTPUT_MP4 = ROOT / f'{RUN_NAME}.mp4'
LAST_FRAME_PATH = ROOT / f'{RUN_NAME}_last_frame.png'
# Show the chosen output name before the expensive GPU operation.
# 在耗時的 GPU 運算前顯示這次的輸出檔名。
print('MODE:', MODE, '| MP4:', OUTPUT_MP4.name)

# Preserve the audio batch-size setting required by the original workflow.
# 保留原始 Workflow 所需的音訊 batch size 設定。
queue_original = engine.queue_prompt

# Normalize audio batch size before submitting the prompt.
# 送出 Prompt 前統一音訊的 batch size。
def queue_with_audio_batch(workflow):
    # Inspect the workflow for audio-latent nodes.
    # 尋找 Workflow 中的 audio latent 節點。
    for node in workflow.values():
        # Keep audio batch size compatible with the video output.
        # 確保音訊 batch size 與影片輸出相容。
        if node.get('class_type') == 'LTXVEmptyLatentAudio':
            # Use one synchronized audio sequence.
            # 使用一組同步音訊。
            node['inputs']['batch_size'] = 1

    # Submit the original workflow after the small compatibility adjustment.
    # 完成相容性調整後，送出原始 Workflow。
    return queue_original(workflow)

# Apply the audio compatibility adjustment to this run.
# 將音訊相容性調整套用至本次生成。
engine.queue_prompt = queue_with_audio_batch

# Pass the selected settings to the original workflow.
# 將設定值傳入原本的 Workflow。
params = {
    'mode': MODE,
    'prompt': PROMPT,
    'width': WIDTH,
    'height': HEIGHT,
    'seconds': SECONDS,
    'fps': FPS,
    'seed': SEED,
    'image_name': image_name,
    'last_image_name': last_image_name,
    'audio_name': engine.ensure_placeholder_audio(),
    'image_strength': IMAGE_STRENGTH,
    'vae_tile_size': VAE_TILE_SIZE,
    'vae_tile_overlap': VAE_TILE_OVERLAP,
    'vae_temporal_size': VAE_TEMPORAL_SIZE,
    'vae_temporal_overlap': VAE_TEMPORAL_OVERLAP,
}

# Check host RAM headroom before beginning a long I2V job.
# 長時間 I2V 生成前先確認主機 RAM 剩餘量。
meminfo = Path('/proc/meminfo').read_text()
# Read the currently available memory without using external packages.
# 不額外安裝套件，直接取得可用記憶體數值。
available_kib = int(next(line.split()[1] for line in meminfo.splitlines() if line.startswith('MemAvailable:')))
# Report RAM availability for memory-pressure troubleshooting.
# 顯示可用 RAM，方便排查記憶體壓力。
print(f'可用 RAM：{available_kib / 1024**2:.1f} GiB', flush=True)
# Warn when I2V memory pressure may terminate the server.
# 當可用記憶體偏低時提醒 I2V 可能造成服務中斷。
if available_kib < 5 * 1024**2:
    # Keep the warning nonfatal because available RAM can change dynamically.
    # RAM 使用量會持續變動，因此僅警告而不直接中止。
    print('注意：可用 RAM 低於 5 GiB，請留意系統記憶體壓力。', flush=True)

# Submit the generation and measure the complete elapsed time.
# 執行生成並測量總耗時。
start_time = time.perf_counter()

# Run the complete workflow and show a short error log if it fails.
# 執行完整 Workflow，失敗時顯示精簡錯誤紀錄。
try:
    # Generate the selected clip with its native two-stage workflow.
    # 使用原生雙階段 Workflow 生成影片。
    generated_path = Path(engine.run_once(RUN_NAME, params, MODEL_TIER))
except Exception:
    # Report the server exit code when it is no longer running.
    # ComfyUI 已中斷時顯示程序結束碼。
    print('ComfyUI exit code:', COMFY_PROC.poll() if 'COMFY_PROC' in globals() else 'unknown')
    # Check Linux memory-controller counters when they are available.
    # 系統支援時讀取 Linux 記憶體控制器計數器。
    events_path = Path('/sys/fs/cgroup/memory.events')
    # Show OOM kill evidence without assuming the process died from OOM.
    # 顯示 OOM Kill 訊號，不直接假定中斷原因。
    if events_path.is_file():
        # Print the counters to distinguish memory kills from other failures.
        # 顯示計數器，以區分記憶體終止與其他錯誤。
        print('Cgroup memory events:\n' + events_path.read_text())
    # Print the relevant end of the ComfyUI server log.
    # 顯示最近的 ComfyUI 日誌內容。
    print(COMFY_LOG.read_text(errors='replace')[-4000:])
    # Preserve the original error and traceback.
    # 保留原始錯誤與 traceback。
    raise

# Copy the resulting MP4 into the stable output path.
# 將生成影片儲存到固定輸出路徑。
shutil.copy2(generated_path, OUTPUT_MP4)

# Report the output location and total workflow time.
# 顯示影片輸出位置與完整生成耗時。
elapsed_seconds = time.perf_counter() - start_time
print('MP4:', OUTPUT_MP4)
print(f'生成耗時：{elapsed_seconds / 60:.2f} 分鐘（{elapsed_seconds:.1f} 秒）')


## 8. 預覽與下載

In [ ]:
# Confirm that the generated MP4 exists.
# 確認生成的 MP4 檔案存在。
assert OUTPUT_MP4.is_file(), '請先執行生成影片 Cell。'

# Preview the video using a positional path compatible with Kaggle IPython.
# 使用 Kaggle IPython 相容的位置參數預覽影片。
display(Video(str(OUTPUT_MP4), embed=True, width=448))

# Provide a direct download link.
# 提供 MP4 下載連結。
display(FileLink(str(OUTPUT_MP4)))


## 9. 擷取最後影格（選用）


In [ ]:
# Count decoded video frames to identify the final frame.
# 計算影片影格數，找出最後一個影格。
import json

# Read the frame count from the saved MP4.
# 從已儲存的 MP4 讀取影格總數。
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(OUTPUT_MP4)],
    capture_output=True, text=True, check=True,
)

# Convert the frame count to a zero-based index.
# 將影格總數轉成從零開始的最後影格索引。
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

# Save the exact last frame as a PNG for the next I2V clip.
# 將最後一個影格儲存成 PNG，供下一段 I2V 使用。
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUTPUT_MP4),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

# Preview and link the extracted image.
# 顯示擷取圖片並提供下載連結。
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))

# Show the input path to use for the next video.
# 顯示下一段影片可直接使用的圖片路徑。
print(f"下一段：MODE = 'i2v'，IMAGE_PATH = '{LAST_FRAME_PATH}'")
